# 面试问题：机器学习数据集应该怎样切分，如何系统发现 Data Leakage？

**一句话回答**：切分单位必须与真实泛化对象一致——同一用户、文档族、设备或患者不能跨集合；有时间因果关系时按 event time 切分并设置 embargo。所有统计量、词表、采样、去重和特征选择只能在 train 拟合，再冻结应用于 validation/test。最后用实体交集、内容指纹、特征可用时间和标签来源做自动审计。

本 Notebook 用标准库与 NumPy 实现随机切分反例、group-stratified split、time split、近似生产可用的泄漏审计和版本 manifest。

In [ ]:
from dataclasses import dataclass
from collections import Counter, defaultdict
import hashlib, json, math, re, unicodedata
import numpy as np

SEED91=9101; rng91=np.random.default_rng(SEED91)
assert SEED91==9101
assert hashlib.sha256(b"train").hexdigest()!=hashlib.sha256(b"test").hexdigest()
assert unicodedata.normalize("NFKC","Ａ")=="A"

## 1. 样本、主体和时间是不同合同

每行样本包含 row ID、group/entity ID、event time、label、文本和特征可用时间。row 是存储单位，group 是统计独立/泛化单位，event time 决定当时能看到什么。随机按 row 切分经常让同一主体同时出现在 train/test。

标签窗口也要明确，例如“未来 7 天是否购买”；切分边界附近需要 purge/embargo，防止 train 标签窗口跨入 test 期间。

In [ ]:
@dataclass(frozen=True)
class Sample91:
    row_id:str; group_id:str; event_time:int; label:int; text:str; feature_available_time:int
    def __post_init__(self):
        if not self.row_id or not self.group_id or self.event_time<0 or self.label not in (0,1) or not self.text or self.feature_available_time<0: raise ValueError("sample_contract")
samples91=[]
group_label91={f"u{i:03d}":int(rng91.random()<.35) for i in range(180)}
for gi,(group,label) in enumerate(group_label91.items()):
    for j in range(4): samples91.append(Sample91(f"{group}-r{j}",group,gi*5+j,label,f"用户 {group} 行为 {j}",gi*5+j))
assert len(samples91)==720 and len({s.row_id for s in samples91})==720
assert len({s.group_id for s in samples91})==180 and set(s.label for s in samples91)=={0,1}
try: Sample91("","g",0,2,"x",0); raise AssertionError("bad sample accepted")
except ValueError as e: assert str(e)=="sample_contract"

## 2. 随机按行切分的泄漏反例

对相邻帧、同一患者多次检查、同一文档多个 chunk 或同一用户行为，row-level random split 会让模型记住主体特征。指标可能很高，但部署到全新主体时崩溃。

下面固定随机排列做 70/15/15，直接计算 group 交集；“先 shuffle 再切”并不自动安全。

In [ ]:
perm91=rng91.permutation(len(samples91)); n_train91=int(.7*len(samples91)); n_val91=int(.15*len(samples91)); naive91={"train":perm91[:n_train91],"val":perm91[n_train91:n_train91+n_val91],"test":perm91[n_train91+n_val91:]}
def groups91(indices): return {samples91[int(i)].group_id for i in indices}
overlap_tv91=groups91(naive91["train"])&groups91(naive91["val"]); overlap_tt91=groups91(naive91["train"])&groups91(naive91["test"])
assert len(overlap_tv91)>50 and len(overlap_tt91)>50
assert sum(map(len,naive91.values()))==len(samples91)
assert set(naive91["train"]).isdisjoint(naive91["test"])

## 3. 手写 group-stratified split

先把 group 作为不可拆原子，按标签分层后分别打乱，再按比例分配 group；最后展开为 row indices。若 group 内标签不一致，需要定义 group label（多数、是否任一阳性）或使用带权 greedy，不能偷偷拆 group。

validation 用于阈值/超参数，test 在最终一次评估前保持封存。

In [ ]:
def group_stratified91(samples,ratios=(.7,.15,.15),seed=91):
    if len(ratios)!=3 or not math.isclose(sum(ratios),1): raise ValueError("ratio_contract")
    by_group=defaultdict(list)
    for i,s in enumerate(samples): by_group[s.group_id].append(i)
    labels={g:{samples[i].label for i in rows} for g,rows in by_group.items()}
    if any(len(v)!=1 for v in labels.values()): raise ValueError("mixed_group_label")
    out={"train":[],"val":[],"test":[]}; local=np.random.default_rng(seed)
    for y in (0,1):
        gs=np.array(sorted(g for g,v in labels.items() if next(iter(v))==y)); local.shuffle(gs); a=round(len(gs)*ratios[0]); b=round(len(gs)*(ratios[0]+ratios[1]))
        for name,part in zip(out,[gs[:a],gs[a:b],gs[b:]]):
            for g in part: out[name].extend(by_group[str(g)])
    return {k:np.array(sorted(v)) for k,v in out.items()}
grouped91=group_stratified91(samples91)
assert not (groups91(grouped91["train"])&groups91(grouped91["val"])) and not (groups91(grouped91["train"])&groups91(grouped91["test"]))
rates91={k:np.mean([samples91[i].label for i in v]) for k,v in grouped91.items()}
assert max(rates91.values())-min(rates91.values())<.04
assert sum(map(len,grouped91.values()))==720

## 4. 时间切分、label horizon 与 embargo

预测未来时应模拟“用过去预测未来”。train event time 小于 cutoff；若 label 使用未来 `H` 天，train 最晚事件必须早于 `val_start-H`。embargo 还能降低同一突发事件在边界两侧的相关性。

时间切分与 group 隔离可能冲突：新用户泛化和未来行为预测是不同问题，应分别做两个 benchmark 或按业务优先级组合。

In [ ]:
def time_split91(samples,train_end,val_end,label_horizon=7,embargo=3):
    train=np.array([i for i,s in enumerate(samples) if s.event_time+label_horizon<train_end-embargo]); val=np.array([i for i,s in enumerate(samples) if train_end<=s.event_time<val_end]); test=np.array([i for i,s in enumerate(samples) if s.event_time>=val_end+embargo]); return {"train":train,"val":val,"test":test}
timed91=time_split91(samples91,600,760,7,3)
assert max(samples91[i].event_time+7 for i in timed91["train"])<597
assert min(samples91[i].event_time for i in timed91["val"])>=600
assert min(samples91[i].event_time for i in timed91["test"])>=763 and set(timed91["train"]).isdisjoint(timed91["test"])

## 5. 内容重复与文档族泄漏

即使 group ID 不同，复制粘贴、模板化文本、图像增强版本仍可能跨集合。exact fingerprint 在统一 normalization 后 hash；near-duplicate 可用 shingle/MinHash。去重必须在切分前先形成 document family，再按 family 切分，而不是删除 test 中“撞到 train”的样本来美化结果。

normalization 规则也需版本化，否则 fingerprint 无法重放。

In [ ]:
def normalize_text91(text): return re.sub(r"\s+"," ",unicodedata.normalize("NFKC",text).strip().casefold())
def fingerprint91(text): return hashlib.sha256(normalize_text91(text).encode()).hexdigest()
train_text91=["退款 流程","模型部署指南","Graph Neural Network"]
test_text91=["退款　流程 ","全新问题","graph neural network"]
train_fp91={fingerprint91(x) for x in train_text91}; leaked_fp91=[x for x in test_text91 if fingerprint91(x) in train_fp91]
assert leaked_fp91==["退款　流程 ","graph neural network"]
assert fingerprint91("Ａ B")==fingerprint91("a b")
assert fingerprint91("ab")!=fingerprint91("a b")

## 6. 预处理只能在 train 拟合

标准化、缺失填补、词表/IDF、特征选择、PCA 和重采样若看过 validation/test，就把其分布信息带入模型。正确接口是 `fit(train)` 产生不可变 state，再 `transform(all splits)`；生产服务加载同一 state。

这里展示 test 有分布漂移时，用全量均值会把 test 悄悄拉回中心，低估真实 skew。

In [ ]:
class Standardizer91:
    def fit(self,x):
        x=np.asarray(x,float); self.mean=x.mean(0); self.std=x.std(0); self.std=np.where(self.std==0,1,self.std); return self
    def transform(self,x): return (np.asarray(x,float)-self.mean)/self.std
x_train91=rng91.normal(0,1,(500,3)); x_test91=rng91.normal(2,1,(200,3)); scaler91=Standardizer91().fit(x_train91); transformed_test91=scaler91.transform(x_test91)
leaked_scaler91=Standardizer91().fit(np.vstack([x_train91,x_test91]))
assert np.allclose(scaler91.transform(x_train91).mean(0),0,atol=1e-12)
assert transformed_test91.mean()>1.7
assert leaked_scaler91.transform(x_test91).mean()<transformed_test91.mean()

## 7. Target leakage 与 feature availability audit

特征值的 event time 不等于它进入系统的 available time。退款结果、人工审核结论、未来聚合可能在预测之后才可用。Point-in-time join 必须要求 `available_time <= prediction_time`；仅检查相关系数发现不了所有泄漏。

建议维护 feature lineage：source、event/available time、窗口和 owner，并在训练快照构建时硬失败。

In [ ]:
@dataclass(frozen=True)
class FeatureValue91:
    entity:str; name:str; value:float; event_time:int; available_time:int
prediction_time91=100
feature_values91=[FeatureValue91("u1","past_clicks",3,90,92),FeatureValue91("u1","refund_result",1,95,105),FeatureValue91("u1","future_7d_count",8,107,108)]
legal91=[f for f in feature_values91 if f.available_time<=prediction_time91]; illegal91=[f for f in feature_values91 if f.available_time>prediction_time91]
assert [f.name for f in legal91]==["past_clicks"]
assert {f.name for f in illegal91}=={"refund_result","future_7d_count"}
assert all(f.event_time<=f.available_time for f in feature_values91)

## 8. Split manifest 与自动门禁

manifest 保存 snapshot、split strategy/seed、group key、time cutoff、horizon/embargo、fingerprint 和 preprocessing state 摘要。CI 检查 row/group/fingerprint 交集、标签分布、时间边界和特征可用性；test 指标必须引用 manifest hash。

这样“换了随机种子涨 2 个点”会留下可审计痕迹，也能准确复现线上回归集。

In [ ]:
split_ids91={k:[samples91[i].row_id for i in v] for k,v in grouped91.items()}; split_sha91=hashlib.sha256(json.dumps(split_ids91,sort_keys=True,separators=(",",":")).encode()).hexdigest(); manifest91={"schema":1,"snapshot":"data-v5","strategy":"group_stratified","group_key":"user_id","seed":91,"split_sha256":split_sha91,"normalizer":"nfkc-casefold-v1"}
digest91=hashlib.sha256(json.dumps(manifest91,sort_keys=True,separators=(",",":")).encode()).hexdigest()
assert len(split_sha91)==len(digest91)==64
assert manifest91["group_key"]=="user_id" and len(split_ids91["test"])>0
assert not (set(split_ids91["train"])&set(split_ids91["test"]))
print({"naive_group_overlap":len(overlap_tt91),"safe_rates":rates91,"sha":digest91[:12]})

## 9. 面试收束、参考与练习

回答闭环：部署泛化对象 → row/group/time 合同 → group stratification → time horizon/embargo → duplicate family → train-only preprocessing → availability lineage → manifest/自动门禁。不要只回答“70/20/10 随机切分”。

练习：实现 group size 不等时的 greedy 平衡；加入 multilabel stratification；用 MinHash 形成文档族；模拟患者跨医院 ID 不一致的实体泄漏。

参考：[scikit-learn 数据泄漏常见陷阱](https://scikit-learn.org/stable/common_pitfalls.html#data-leakage)、[Purged/Embargo 时间验证思想](https://papers.ssrn.com/sol3/papers.cfm?abstract_id=3257419)、[Google Rules of ML](https://developers.google.com/machine-learning/guides/rules-of-ml)。核心切分代码已在上面从零实现。